# 05 — E4 (T-arm), E5 (robustness audit incl. model swaps), scale ablation
E5 groups by served model — restart vLLM between groups as printed.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
print("project root:", ROOT)

In [ ]:
import json, yaml
from traitmix.runner import run_grid
from traitmix.classifier import make_scorer
MAN = json.loads((ROOT / "configs" / "MANIFEST.json").read_text()); scorer = make_scorer()

In [ ]:
# E4 — requires the LoRA-enabled server (see notebook 03)
rows = run_grid([ROOT / "configs" / c for c in MAN["E4"]["configs"]], MAN["E4"]["seeds"], trait_scorer=scorer)

In [ ]:
# E5 — run in model groups; the registry makes re-runs after each server restart free.
by_model = {}
for c in MAN["E5"]["configs"]:
    m = yaml.safe_load(open(ROOT / "configs" / c)).get("llm", {}).get("model", "meta-llama/Llama-3.1-8B-Instruct")
    by_model.setdefault(m, []).append(c)
for model, cfgs in by_model.items():
    print(f"\n### Serve this model, then continue: vllm serve {model}\n({len(cfgs)} configs)")
    rows = run_grid([ROOT / "configs" / c for c in cfgs], MAN["E5"]["seeds"], trait_scorer=scorer)

In [ ]:
rows = run_grid([ROOT / "configs" / c for c in MAN["SCALE"]["configs"]], MAN["SCALE"]["seeds"], trait_scorer=scorer)